In [ ]:
import pandas as pd
import numpy as np
import os
import math
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
import warnings
warnings.simplefilter("ignore")

In [ ]:
if "test_chars_rank_imputed" not in os.listdir():
    os.mkdir("test_chars_rank_imputed")

if "test_chars_raw_no_impute" not in os.listdir():
    os.mkdir("test_chars_raw_no_impute")

if "test_chars_raw_no_impute_decile_bar" not in os.listdir():
    os.mkdir("test_chars_raw_no_impute_decile_bar")

if "test_chars_raw_no_impute_decile_only_mean" not in os.listdir():
    os.mkdir("test_chars_raw_no_impute_decile_only_mean")

if "test_chars_raw_no_impute_decile_box_plot" not in os.listdir():
    os.mkdir("test_chars_raw_no_impute_decile_box_plot")

In [ ]:
# char_list = ['maxret', 'grltnoa', 'pchcurrat', 'abr', 'pm', 'pchsale_pchinvt',
#        'salecash', 'herf', 'cashdebt', 'sp', 'std_turn', 'rdm', 'roic',
#        'chinv', 'adm', 'acc', 'me', 'tang', 'pchgm_pchsale', 'absacc',
#        'pchsale_pchrect', 'rvar_ff3', 'tb', 'chempia', 'mom1m', 'invest',
#        'roe', 'convind', 'std_dolvol', 'op', 'cfp_ia', 'seas1a',
#        'secured', 'chpmia', 'pscore', 'quick', 'baspread', 'mom6m',
#        'pctacc', 'rd', 'dy', 'realestate', 'chatoia', 're', 'agr', 'divi',
#        'ni', 'rna', 'nincr', 'chcsho', 'indmom', 'egr', 'salerec', 'roa',
#        'cinvest', 'pchdepr', 'gma', 'securedind', 'currat', 'divo', 'cfp',
#        'ato', 'cashpr', 'ill', 'beta', 'roavol', 'turn', 'grcapx',
#        'pchquick', 'ep', 'me_ia', 'zerotrade', 'rvar_mean', 'depr',
#        'pchsaleinv', 'hire', 'stdcf', 'chtx', 'rd_sale', 'saleinv',
#        'rsup', 'rvar_capm', 'bm_ia', 'mom36m', 'pchsale_pchxsga', 'alm',
#        'mom12m', 'pchcapx_ia', 'sgr', 'sue', 'lev', 'age', 'mom60m', 'bm',
#        'sin', 'cash', 'dolvol', 'lgr', 'noa', 'stdacc', 'chmom']
char_list = ['cinvest', 'rvar_capm', 'mom36m', 'zerotrade', 'rna', 'chtx', 'pm',
       'depr', 'cash', 'me_ia', 'adm', 'bm_ia', 'cashdebt',
       'mom12m', 'baspread', 'sp', 'mom1m', 'rvar_ff3', 'ep', 'dy', 're',
       'nincr', 'rdm', 'rsup', 'lgr', 'chpmia', 'std_dolvol', 'rd_sale',
       'beta', 'rvar_mean', 'mom60m', 'chcsho', 'roa', 'acc', 'ato',
       'sue', 'op', 'alm', 'noa', 'mom6m', 'me', 'cfp', 'pscore',
       'seas1a', 'roe', 'maxret', 'std_turn', 'sgr', 'grltnoa', 'gma',
       'ni', 'dolvol', 'bm', 'pctacc', 'herf', 'lev', 'agr', 'ill', 'abr',
       'hire', 'turn']
char_list = sorted(char_list)
rank_columns = [f"rank_{col}" for col in char_list]

In [ ]:
df_raw_no_impute = pd.read_feather("chars/chars_raw_no_impute.feather")
df_raw_no_impute['date'] = pd.to_datetime(df_raw_no_impute['date']).dt.strftime('%Y-%m-%d')
print(df_raw_no_impute.shape)

df_raw_imputed = pd.read_feather("chars/chars_raw_imputed.feather")
df_raw_imputed['date'] = pd.to_datetime(df_raw_imputed['date']).dt.strftime('%Y-%m-%d')
print(df_raw_imputed.shape)

df_rank_no_impute = pd.read_feather("chars/chars_rank_no_impute.feather")
df_rank_no_impute['date'] = pd.to_datetime(df_rank_no_impute['date']).dt.strftime('%Y-%m-%d')
print(df_rank_no_impute.shape)

df_rank_imputed = pd.read_feather("chars/chars_rank_imputed.feather")
df_rank_imputed['date'] = pd.to_datetime(df_rank_imputed['date']).dt.strftime('%Y-%m-%d')
print(df_rank_imputed.shape)

#### Filter out 61 chars version

In [ ]:
df_raw_no_impute = df_raw_no_impute[['gvkey', 'permno', 'sic', 'ret', 'exchcd', 'shrcd', 'ticker', 'conm', 'comnam', 'date', 'prc', 'shrout'] + char_list]
df_raw_no_impute.to_feather("chars60_raw_no_impute.feather")

In [ ]:
df_raw_imputed = df_raw_imputed[['gvkey', 'permno', 'sic', 'ret', 'exchcd', 'shrcd', 'ticker', 'conm', 'comnam', 'date','ffi49', 'prc', 'shrout'] + char_list]
df_raw_imputed.to_feather("chars60_raw_imputed.feather")

In [ ]:
df_rank_no_impute = df_rank_no_impute[['gvkey', 'permno', 'sic', 'ret', 'exchcd', 'shrcd', 'date', 'lag_me', 'ticker', 'conm', 'comnam', 'prc', 'shrout', 'log_me'] + rank_columns]
df_rank_no_impute.to_feather("chars60_rank_no_impute.feather")

In [ ]:
df_rank_imputed = df_rank_imputed[['gvkey', 'permno', 'sic', 'ret', 'exchcd', 'shrcd', 'date', 'lag_me', 'ticker', 'conm', 'comnam', 'prc', 'shrout', 'log_me'] + rank_columns]
df_rank_imputed.to_feather("chars60_rank_imputed.feather")

In [ ]:
df_raw_no_impute[df_raw_no_impute['date'] >= '2014-05-01']['bm'].describe()

#### Start to check our data

In [ ]:
all_dates = sorted(list(df_rank_imputed['date'].unique()))

##### Draw rank imputed chars histogram

In [ ]:
subplot_x = math.ceil(len(char_list) / 5)

def plot_for_date(this_date):
    select_char = df_rank_imputed[df_rank_imputed['date'] == this_date]

    # plot all the char
    fig, axs = plt.subplots(subplot_x, 5, figsize=(20, int(3*subplot_x)))
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

    for i, column in enumerate(rank_columns):
        ax = axs[i // 5, i % 5]
        ax.hist(select_char[column], bins=20, range=(-1, 1), edgecolor='black')
        ax.set_title(f'char: {column}')
        ax.set_xlabel('Values')
        ax.set_ylabel('Frequency')
    plt.savefig(f"test_chars_rank_imputed/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
    plt.show()
    plt.close()

Parallel(n_jobs=50)(delayed(plot_for_date)(this_date) for this_date in all_dates)

##### Draw raw no-impute histogram

In [ ]:
subplot_x = math.ceil(len(char_list) / 5)

def plot_for_date(this_date):
    select_char = df_raw_no_impute[df_raw_no_impute['date'] == this_date]

    # plot all the char
    fig, axs = plt.subplots(subplot_x, 5, figsize=(20, int(3*subplot_x)))
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

    for i, column in enumerate(char_list):
        non_na_values = select_char[column].dropna()
        if not non_na_values.empty:
            min_value = non_na_values.min()
            max_value = non_na_values.max()
        else:
            min_value = -1
            max_value = 1
        
        ax = axs[i // 5, i % 5]
        # ax.hist(select_char[column], bins=20, range=(min_value, max_value), edgecolor='black')
        ax.hist(select_char[column].dropna(), bins=20, range=(min_value, max_value), edgecolor='black')
        ax.set_title(f'char: {column}')
        ax.set_xlabel('Values')
        ax.set_ylabel('Frequency')
    plt.savefig(f"test_chars_raw_no_impute/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
    plt.show()
    plt.close()

Parallel(n_jobs=50)(delayed(plot_for_date)(this_date) for this_date in all_dates)

##### Draw raw no-impute decile barplot

In [ ]:
# df_temp = df_raw_no_impute[df_raw_no_impute['date'] == '1950-11-30'].reset_index(drop=True)

# char = 'hire'
# this_char = df_temp[[char]].dropna().reset_index(drop=True)
# print(f"Char {char}: {this_char.shape}")

# if len(this_char) < 10:
#     print("No data for this char")
# else:
#     this_char['decile'] = pd.qcut(this_char[char], 10, labels=False, duplicates='drop')
#     if len(this_char['decile'].unique()) == 1:
#         this_char['decile'] = 1
#     decile_stats = this_char.groupby('decile')[char].agg(['mean', 'min', 'max']).reset_index()

#     print(decile_stats)

In [ ]:
subplot_x = math.ceil(len(char_list) / 3)

def plot_for_date(this_date):
    select_char = df_raw_no_impute[df_raw_no_impute['date'] == this_date].reset_index(drop=True)

    # plot all the char
    fig, axs = plt.subplots(subplot_x, 3, figsize=(20, int(4*subplot_x)))
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

    for i, column in enumerate(char_list):
        non_na_values = select_char[[column]].dropna().reset_index(drop=True)
        if len(non_na_values) >= 10:
            # 将数据分为10个decile，如果唯一值数量少于10，则分为唯一值数量的分位数
            num_bins = 10
            non_na_values['decile'] = pd.qcut(non_na_values[column], num_bins, labels=False, duplicates='drop') + 1
            if len(non_na_values['decile'].unique()) == 1:
                non_na_values['decile'] = 1

            # 计算每个decile的均值、最小值和最大值
            decile_stats = non_na_values.groupby('decile')[column].agg(['mean', 'min', 'max']).reset_index()
            print(f"date: {this_date}, char: {column}, decile: {decile_stats['decile'].unique()}")

            # 绘制散点图
            ax = axs[i // 3, i % 3]
            ax.scatter(decile_stats['decile'], decile_stats['mean'], label='Mean', color='blue', marker='o')
            ax.scatter(decile_stats['decile'], decile_stats['min'], label='Min', color='red', marker='o')
            ax.scatter(decile_stats['decile'], decile_stats['max'], label='Max', color='green', marker='o')
            ax.set_title(f'char: {column}')
            ax.set_xlabel('Decile')
            ax.set_ylabel('Value')
            max_decile = int(decile_stats['decile'].max())
            ax.set_xticks(range(1, max_decile + 1))
            ax.set_xticklabels(range(1, max_decile + 1))
            ax.legend()
        else:
            continue

    plt.savefig(f"test_chars_raw_no_impute_decile_bar/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
    plt.show()
    plt.close()

Parallel(n_jobs=50)(delayed(plot_for_date)(this_date) for this_date in all_dates)

In [ ]:
subplot_x = math.ceil(len(char_list) / 3)

def plot_for_date(this_date):
    select_char = df_raw_no_impute[df_raw_no_impute['date'] == this_date].reset_index(drop=True)

    # plot all the char
    fig, axs = plt.subplots(subplot_x, 3, figsize=(20, int(4*subplot_x)))
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

    for i, column in enumerate(char_list):
        non_na_values = select_char[[column]].dropna().reset_index(drop=True)
        if len(non_na_values) >= 10:
            # 将数据分为10个decile，如果唯一值数量少于10，则分为唯一值数量的分位数
            num_bins = 10
            non_na_values['decile'] = pd.qcut(non_na_values[column], num_bins, labels=False, duplicates='drop') + 1
            if len(non_na_values['decile'].unique()) == 1:
                non_na_values['decile'] = 1

            # 计算每个decile的均值、最小值和最大值
            decile_stats = non_na_values.groupby('decile')[column].agg(['mean', 'min', 'max']).reset_index()
            print(f"date: {this_date}, char: {column}, decile: {decile_stats['decile'].unique()}")

            # 绘制散点图
            ax = axs[i // 3, i % 3]
            ax.scatter(decile_stats['decile'], decile_stats['mean'], label='Mean', color='red', marker='o')
            ax.set_title(f'char: {column}')
            ax.set_xlabel('Decile')
            ax.set_ylabel('Value')
            max_decile = int(decile_stats['decile'].max())
            ax.set_xticks(range(1, max_decile + 1))
            ax.set_xticklabels(range(1, max_decile + 1))
            ax.legend()
        else:
            continue

    plt.savefig(f"test_chars_raw_no_impute_decile_only_mean/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
    plt.show()
    plt.close()

Parallel(n_jobs=50)(delayed(plot_for_date)(this_date) for this_date in all_dates)

##### Draw raw no-impute box plot

In [ ]:
subplot_x = math.ceil(len(char_list) / 3)

def plot_for_date(this_date):
    select_char = df_raw_no_impute[df_raw_no_impute['date'] == this_date].reset_index(drop=True)

    # plot all the char
    fig, axs = plt.subplots(subplot_x, 3, figsize=(20, int(4*subplot_x)))
    fig.subplots_adjust(hspace=0.5, wspace=0.5)

    for i, column in enumerate(char_list):
        non_na_values = select_char[[column]].dropna().reset_index(drop=True)
        if len(non_na_values) >= 10:
            # 将数据分为10个decile，如果唯一值数量少于10，则分为唯一值数量的分位数
            num_bins = 10
            non_na_values['decile'] = pd.qcut(non_na_values[column], num_bins, labels=False, duplicates='drop') + 1
            if len(non_na_values['decile'].unique()) == 1:
                non_na_values['decile'] = 1

            # 计算每个decile的均值、最小值和最大值
            decile_stats = non_na_values.groupby('decile')[column].agg(['mean', 'min', 'max']).reset_index()
            print(f"date: {this_date}, char: {column}, decile: {decile_stats['decile'].unique()}")

            # 绘制箱线图
            ax = axs[i // 3, i % 3]
            non_na_values.boxplot(column=column, by='decile', ax=ax)
            ax.set_title(f'char: {column}')
            ax.set_xlabel('Decile')
            ax.set_ylabel('Value')
            max_decile = int(decile_stats['decile'].max())
            ax.set_xticks(range(1, max_decile + 1))
            ax.set_xticklabels(range(1, max_decile + 1))
        else:
            continue

    plt.suptitle(f'Boxplots for {this_date}')
    plt.savefig(f"test_chars_raw_no_impute_decile_box_plot/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
    plt.show()
    plt.close()

Parallel(n_jobs=50)(delayed(plot_for_date)(this_date) for this_date in all_dates)

In [ ]:
subplot_x = math.ceil(len(char_list) / 3)

this_date = '2024-12-31'

select_char = df_raw_no_impute[df_raw_no_impute['date'] == this_date].reset_index(drop=True)

# plot all the char
fig, axs = plt.subplots(subplot_x, 3, figsize=(20, int(4*subplot_x)))
fig.subplots_adjust(hspace=0.5, wspace=0.5)

for i, column in enumerate(char_list):
    non_na_values = select_char[[column]].dropna().reset_index(drop=True)
    if len(non_na_values) >= 10:
        # 将数据分为10个decile，如果唯一值数量少于10，则分为唯一值数量的分位数
        num_bins = 10
        non_na_values['decile'] = pd.qcut(non_na_values[column], num_bins, labels=False, duplicates='drop') + 1
        if len(non_na_values['decile'].unique()) == 1:
            non_na_values['decile'] = 1

        # 计算每个decile的均值、最小值和最大值
        non_na_values = non_na_values[non_na_values['decile'] != 1].reset_index(drop=True)
        non_na_values = non_na_values[non_na_values['decile'] != 10].reset_index(drop=True)
        decile_stats = non_na_values.groupby('decile')[column].agg(['mean', 'min', 'max']).reset_index()
        # print(f"date: {this_date}, char: {column}, decile: {decile_stats['decile'].unique()}")

        # 绘制箱线图
        ax = axs[i // 3, i % 3]
        non_na_values.boxplot(column=column, by='decile', ax=ax)
        ax.set_title(f'char: {column}')
        ax.set_xlabel('Decile')
        ax.set_ylabel('Value')
        max_decile = int(decile_stats['decile'].max())
        ax.set_xticks(range(1, max_decile))
        ax.set_xticklabels(range(2, max_decile+1))
    else:
        continue

plt.suptitle(f'Boxplots for {this_date}')
# plt.savefig(f"test_chars_raw_no_impute_decile_box_plot/char_plot_{str(this_date)[0:4]}_{str(this_date)[5:7]}.png")
plt.show()
plt.close()

### 2025-07-06 updates (Yunting Liu)
1. review the data structure
2. compare SIZ with CIZ

**SIZ structure**

In [ ]:
import pandas as pd

# Read annual accounting characteristics data
chars_a = pd.read_feather('SIZdata/chars_a_accounting.feather')
print("\nAnnual Accounting Data Structure:")
print(chars_a.info())
print("\nFirst 5 rows of Annual Data:")
print(chars_a.head())

print("\n" + "="*50 + "\n")

# Read quarterly accounting characteristics data
chars_q = pd.read_feather('SIZdata/chars_q_accounting.feather')
print("\nQuarterly Accounting Data Structure:")
print(chars_q.info())
print("\nFirst 5 rows of Quarterly Data:")
print(chars_q.head())

In [ ]:
import pandas as pd

# Read annual accounting characteristics data
chars_a = pd.read_feather('chars_a_accounting.feather')
print("\nAnnual Accounting Data Structure:")
print(chars_a.info())
print("\nFirst 5 rows of Annual Data:")
print(chars_a.head())

print("\n" + "="*50 + "\n")

# Read quarterly accounting characteristics data
chars_q = pd.read_feather('chars_q_accounting.feather')
print("\nQuarterly Accounting Data Structure:")
print(chars_q.info())
print("\nFirst 5 rows of Quarterly Data:")
print(chars_q.head())

In [ ]:
chars_q

#### **CRSP sql structure**

In [ ]:
import pandas as pd
import wrds
db = wrds.Connection(wrds_username='phd22jm', wrds_password='jmwarwickap1998!')

In [ ]:
# pd.DataFrame(db.list_libraries())
pd.DataFrame(db.list_tables(library="crsp"))

In [ ]:
db.describe_table(library="crsp", table="metasiztociz")

In [ ]:
data = db.get_table(library="crsp", table="metasiztociz")
data

### 2025-07-13 updates
1. add permno = 14593
2. replace date(1959-01-01) with date(2023-01-01)
3. add wrds user name and pssword
4. add f.close() and conn.close()
5. compare version1.0 with version2.0

In [ ]:
import pandas as pd

#### abr.py

In [ ]:
# version 1.0
abr = pd.read_feather('SIZdata/abr.feather')
print(abr.info())
print(abr.head())

In [ ]:
# version 2.0
abr = pd.read_feather('abr.feather')
abr

#### beta.py

In [ ]:
# Version 1.0
beta = pd.read_feather('SIZdata/beta.feather')
beta[beta['permno'] == 14593]

In [ ]:
# Version 2.0
beta = pd.read_feather('beta.feather')
beta

#### bid_ask_spread.py

In [ ]:
# Version 1.0
baspread = pd.read_feather('SIZdata/baspread.feather')
baspread

In [ ]:
# Version 2.0
baspread = pd.read_feather('baspread.feather')
baspread

#### iclink_ciz.sas

In [ ]:
# Version 1.0
iclink = pd.read_feather('SIZdata/iclink.feather')
iclink

In [ ]:
import pandas as pd

df = pd.read_csv('iclink_ciz.csv')
df.columns = [col.lower() for col in df.columns]

df.to_feather('iclink_ciz.feather')

In [ ]:
# Version 2.0
iclink = pd.read_feather('iclink_ciz.feather')
iclink

#### ill.py

In [ ]:
# Version 1.0
ill = pd.read_feather('SIZdata/ill.feather')
ill

In [ ]:
# Version 2.0
ill = pd.read_feather('ill.feather')
ill

#### maxret_d.py

In [ ]:
# Version 1.0
maxret = pd.read_feather('SIZdata/maxret.feather')
maxret

In [ ]:
# Version 2.0
maxret = pd.read_feather('maxret.feather')
maxret

#### myre.py

In [ ]:
# Version 1.0
myre = pd.read_feather('SIZdata/myre.feather')
myre

In [ ]:
# Version 2.0
myre = pd.read_feather('myre.feather')
myre

#### rvar_capm.py

In [ ]:
# Version 1.0
rvar_capm = pd.read_feather('SIZdata/rvar_capm.feather')
rvar_capm

In [ ]:
# Version 2.0
rvar_capm = pd.read_feather('rvar_capm.feather')
rvar_capm

#### rvar_ff3.py

In [ ]:
# Version 1.0
rvar_ff3 = pd.read_feather('SIZdata/rvar_ff3.feather')
rvar_ff3

In [ ]:
# Version 2.0
rvar_ff3 = pd.read_feather('rvar_ff3.feather')
rvar_ff3

#### rvar_mean.py

In [ ]:
# Version 1.0
rvar_mean = pd.read_feather('SIZdata/rvar_mean.feather')
rvar_mean

In [ ]:
# Version 2.0
rvar_mean = pd.read_feather('rvar_mean.feather')
rvar_mean

#### std_dolvol.py

In [ ]:
# Version 1.0
std_dolvol = pd.read_feather('SIZdata/std_dolvol.feather')
std_dolvol

In [ ]:
# Version 2.0
std_dolvol = pd.read_feather('std_dolvol.feather')
std_dolvol

#### std_turn.py

In [ ]:
# Version 1.0
std_turn = pd.read_feather('SIZdata/std_turn.feather')
std_turn

In [ ]:
# Version 2.0
std_turn = pd.read_feather('std_turn.feather')
std_turn

#### sue.py

In [ ]:
# Version 1.0
sue = pd.read_feather('SIZdata/sue.feather')
sue

In [ ]:
# Version 2.0
sue = pd.read_feather('sue.feather')
sue

#### zerotrade.py

In [ ]:
# Version 1.0
zerotrade = pd.read_feather('SIZdata/zerotrade.feather')
zerotrade

In [ ]:
# Version 2.0
zerotrade = pd.read_feather('zerotrade.feather')
zerotrade

#### merge_chars.py

In [ ]:
# Version 1.0
chars_a_raw = pd.read_feather('SIZdata/chars_a_raw.feather')
chars_a_raw

In [ ]:
# Version 1.0
chars_q_raw = pd.read_feather('SIZdata/chars_q_raw.feather')
chars_q_raw

In [ ]:
# Version 2.0
chars_a_raw = pd.read_feather('chars_a_raw.feather')
chars_a_raw

In [ ]:
# Version 2.0
chars_q_raw = pd.read_feather('chars_q_raw.feather')
chars_q_raw

#### impute_rank_output_bchmk.py

In [ ]:
# Read rank and raw data
chars_rank_imputed = pd.read_feather('chars_rank_imputed.feather')
chars_rank_no_impute = pd.read_feather('chars_rank_no_impute.feather')
chars_raw_imputed = pd.read_feather('chars_raw_imputed.feather')
chars_raw_no_impute = pd.read_feather('chars_raw_no_impute.feather')

In [ ]:
chars_rank_imputed[chars_rank_imputed['permno']==14593]

In [ ]:
chars_rank_no_impute[chars_rank_no_impute['permno']==14593]

In [ ]:
chars_raw_imputed[chars_raw_imputed['permno']==14593]

In [ ]:
chars_raw_no_impute[chars_raw_no_impute['permno']==14593]

### 2025-07-14 updates

1. Compare chars-version2.0 with chars-version1.0

In [ ]:
import pandas as pd
import numpy as np

new_file = "/home/yuntingliu/myHub/EquityChars/"
old_file = "/home/yuntingliu/myHub/chars60/"

files_name = [
    'chars60_rank_imputed.feather',
    'chars60_rank_no_impute.feather',
    'chars60_raw_imputed.feather',
    'chars60_raw_no_impute.feather'
]

keys = ['gvkey', 'permno', 'sic', 'date', 'comnam']

for file_name in files_name:
    print('='*10, file_name, '='*10)
    new = pd.read_feather(new_file + file_name)
    old = pd.read_feather(old_file + file_name)
    new = new[new['permno'] == 14593].reset_index(drop=True)
    old = old[old['permno'] == 14593].reset_index(drop=True)

    merged = pd.merge(
        new, 
        old, 
        on=keys, 
        suffixes=('_new', '_old')
    )

    num_cols = [col for col in new.select_dtypes(include=[np.number]).columns if col not in keys]
    diff = merged[[k for k in keys]]
    for col in num_cols:
        diff[col + '_diff'] = merged[col + '_new'] - merged[col + '_old']

    print(diff.describe())

    diff.describe().to_csv(f"diff_{file_name.replace('.feather', '.csv')}", index=True)